# Parallelization with Langsmith API

Steps:
Have Agent.py file ready in the studio folder
keep .env file with LangSmith API keys and GOOGle API keys ready
set langgraph.json file ready
To start the local development server, run the following command in your terminal in the /studio directory in this module: BAsh ' langgraph dev '
check for output:
API: http://127.0.0.1:2024

🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
📚 API Docs: http://127.0.0.1:2024/docs
This in-memory server is designed for development and testing. For production use, please use LangSmith Deployment.

Copy url = http://127.0.0.1:2024
Note:
check for more info : https://docs.langchain.com/langsmith/quick-start-studio#local-development-server
check for LangSmith setup guide in the begining of the course.
Connecting to the Local Server Using the SDK
You use the LangGraph SDK to talk to your local agent: python

from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client = get_client(url=url)
This client object lets you:

- list assistants
- create threads
- run your agent
- stream results
Create a thread for storing event checkpoints?¶
A thread is a conversation session.

You run your agent like this: python

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["tools"],
):
Note: We can add interrupt_before=["node"] when compiling the graph that is running in Studio or with the API, you can also pass interrupt_before to the stream method directly.

What happens?
Your agent runs step by step:

- LLM node
- breakpoint to check with user
- Tool node
- LLM node
- Final answer
After each step, LangGraph sends you a chunk containing the updated state.

# Goal:  Running parallel loops in LangSmith studio and on Development framework



In [1]:
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client  = get_client(url=url)

# Search all hosted graphs
assistants = await client.assistants.search()
# list the agents
assistants

[{'assistant_id': '9ed195fc-c808-53c8-8728-542eb51a4833',
  'graph_id': 'parallelization_with_LLM',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'parallelization_with_LLM',
  'created_at': '2026-05-08T15:56:28.576919+00:00',
  'updated_at': '2026-05-08T15:56:28.576919+00:00',
  'version': 1,
  'description': None}]

In [2]:
# create a thread
thread = await client.threads.create()
print(thread)

{'thread_id': '019e0858-aac4-75a3-a86a-1da0176609be', 'created_at': '2026-05-08T16:08:09.156040+00:00', 'updated_at': '2026-05-08T16:08:09.156040+00:00', 'state_updated_at': '2026-05-08T16:08:09.156040+00:00', 'metadata': {}, 'status': 'idle', 'config': {}, 'values': None}


In [3]:
# run agent parallelization_with_LLM with input question 

Question = " Tell me about Lord Krishnain few words"

# run 
chunks =[]
async for chunk in client.runs.stream(thread['thread_id'],
                                      assistant_id="parallelization_with_LLM",
                                      input={ 'Question': Question},
                                      stream_mode='values') :
    chunks.append(chunk)
    print(chunk)
    print("-"*40)

StreamPart(event='metadata', data={'run_id': '019e085f-c3d9-7fa0-b33d-7e80628ca3ff', 'attempt': 1}, id=None)
----------------------------------------
StreamPart(event='values', data={'Question': ' Tell me about Lord Krishna', 'summery_context': []}, id=None)
----------------------------------------
StreamPart(event='values', data={'Question': ' Tell me about Lord Krishna', 'summery_context': []}, id=None)
----------------------------------------
StreamPart(event='values', data={'Question': ' Tell me about Lord Krishna', 'summery_context': ["<Document Title: Krishna\n URL: https://en.wikipedia.org/wiki/Krishna\n content: Krishna (; Sanskrit: कृष्ण, IAST: Kṛṣṇa Sanskrit: [ˈkr̩ʂɳɐ] )  is a major deity in Hinduism. He is worshipped as the eighth avatar of Vishnu and also as the Supreme God in his own right. He is the god of protection, compassion, tenderness, and love; and is widely revered among Hindu divinities. Krishna's birthday is celebrated every year by Hindus on Krishna Janmashtami

In [7]:
Answer = chunks[-1].data.get('Answer', None)['content']
Answer

'Lord Krishna is a central and revered deity in Hinduism, embodying a complex and multifaceted divine personality. He is widely worshipped as the **eighth avatar of Vishnu**, the preserver god, and is also revered as the **Supreme God (Svayam Bhagavan)** in his own right, particularly within traditions like Krishnaism.\n\nHere\'s a detailed look at Lord Krishna:\n\n*   **Identity and Attributes:**\n    *   He is considered the **Supreme Personality of Godhead**, an historical figure who appeared on Earth approximately 5,000 years ago.\n    *   Krishna is the god of **protection, compassion, tenderness, and love**, symbolizing these virtues to an unlimited degree.\n    *   He is described as "all-attractive" and possesses all six divine opulences perfectly. His knowledge is boundless, remembering all past appearances.\n\n*   **Life and Narratives (Krishna Līlā):**\n    *   Born in **Mathura** to Devaki and Vasudeva, his life story is filled with divine exploits.\n    *   To escape the t